# Task 2: Insurance Data Cleaning\n\nClean the raw insurance dataset for wildfire risk prediction.\n\n**Steps:**\n1. Load & explore\n2. Drop columns with >25% missing\n3. Create binary target (fire_occurred: yes/no)\n4. Handle remaining missing values\n5. Balance classes via undersampling\n6. Save cleaned data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/insurance_data.csv", low_memory=False)
print("Shape:", df.shape)
df.head()

Shape: (47033, 76)


,Year,ZIP,Avg Fire Risk Score,Avg PPC,CAT Cov A Fire - Incurred Losses,CAT Cov A Fire - Number of Claims,CAT Cov A Smoke - Incurred Losses,CAT Cov A Smoke - Number of Claims,CAT Cov C Fire - Incurred Losses,CAT Cov C Fire - Number of Claims,...,average_household_size,educational_attainment_bachelor_or_higher,poverty_status,housing_occupancy_number,housing_value,year_structure_built,housing_vacancy_number,median_monthly_housing_costs,owner_occupied_housing_units,renter_occupied_housing_units
0,2018,90003,1.000,1.000,0,0,0,0,0,0,...,4.03,2294.0,18743.0,18349.0,547600.0,18349.0,715.0,1609.0,4692.0,12942.0
1,2018,90004,0.420,1.075,0,0,0,0,0,0,...,2.44,11822.0,10994.0,26046.0,1457200.0,26046.0,2311.0,1847.0,4011.0,19724.0
2,2018,90005,0.510,1.060,0,0,0,0,0,0,...,2.18,7582.0,9463.0,19357.0,1084400.0,19357.0,1880.0,1651.0,1572.0,15905.0
3,2018,90006,0.605,1.040,0,0,0,0,0,0,...,2.79,6758.0,13274.0,21475.0,841900.0,21475.0,2207.0,1415.0,1951.0,17317.0
4,2018,90007,0.000,2.000,0,0,0,0,0,0,...,2.75,3536.0,13473.0,14153.0,852900.0,14153.0,1702.0,1504.0,1297.0,11154.0


In [2]:
# Check missing % per column
missing_pct = df.isna().mean().sort_values(ascending=False)
print("Missing % per column:\n")
print(missing_pct.to_string())

Missing % per column:

FIRE_NUM                                     1.000000
COMPLEX_ID                                   0.998724
COMPLEX_NA                                   0.997704
COMMENTS                                     0.956754
INC_NUM                                      0.834031
CONT_DATE                                    0.832692
UNIT_ID                                      0.832118
FIRE_NAME                                    0.831756
station                                      0.831714
avg_tmax_c                                   0.831714
avg_tmin_c                                   0.831714
tot_prcp_mm                                  0.831714
Alarm_Date2                                  0.831374
year_month                                   0.831374
ALARM_DATE                                   0.831374
C_METHOD                                     0.831267
AGENCY_ID                                    0.831012
GIS_ACRES                                    0.831012
CAUSE

## Step 2: Drop Columns with >25% Missing

In [3]:
# Drop columns with >25% missing
threshold = 0.25
cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns (>25% missing):")
for c in cols_to_drop:
    print(f"  {c}: {missing_pct[c]:.1%}")

df = df.drop(columns=cols_to_drop)
print(f"\nRemaining shape: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Dropping 29 columns (>25% missing):
  FIRE_NUM: 100.0%
  COMPLEX_ID: 99.9%
  COMPLEX_NA: 99.8%
  COMMENTS: 95.7%
  INC_NUM: 83.4%
  CONT_DATE: 83.3%
  UNIT_ID: 83.2%
  FIRE_NAME: 83.2%
  station: 83.2%
  avg_tmax_c: 83.2%
  avg_tmin_c: 83.2%
  tot_prcp_mm: 83.2%
  Alarm_Date2: 83.1%
  year_month: 83.1%
  ALARM_DATE: 83.1%
  C_METHOD: 83.1%
  AGENCY_ID: 83.1%
  GIS_ACRES: 83.1%
  CAUSE: 83.1%
  FIRE_NAME_ID: 83.1%
  AGENCY: 83.1%
  OBJECTID: 83.1%
  longitude: 83.1%
  latitude: 83.1%
  Shape__Len: 83.1%
  Shape__Are: 83.1%
  DECADES: 83.1%
  OBJECTIVE: 83.1%
  Avg PPC: 57.7%

Remaining shape: (47033, 47)
Remaining columns: ['Year', 'ZIP', 'Avg Fire Risk Score', 'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims', 'CAT Cov A Smoke -  Incurred Losses', 'CAT Cov A Smoke -  Number of Claims', 'CAT Cov C Fire -  Incurred Losses', 'CAT Cov C Fire -  Number of Claims', 'CAT Cov C Smoke -  Incurred Losses', 'CAT Cov C Smoke -  Number of Claims', 'Cov A Amount Weighted Avg'

## Step 3: Create Binary Target Variable

In [4]:
# Create binary target: did a fire occur?
# Based on any fire claims (CAT or Non-CAT) being > 0
fire_claims_cols = [
    "CAT Cov A Fire -  Number of Claims",
    "CAT Cov C Fire -  Number of Claims",
    "Non-CAT Cov A Fire -  Number of Claims",
    "Non-CAT Cov C Fire -  Number of Claims",
]

# Check these columns exist
for col in fire_claims_cols:
    assert col in df.columns, f"Missing column: {col}"

df["fire_occurred"] = np.where(
    df[fire_claims_cols].sum(axis=1) > 0, "yes", "no"
)

print("Target distribution:")
print(df["fire_occurred"].value_counts())

Target distribution:
fire_occurred
no     35527
yes    11506
Name: count, dtype: int64


## Step 4: Handle Remaining Missing Values

In [5]:
# Check remaining missing values
remaining_missing = df.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0]
print(f"Columns with remaining missing values: {len(remaining_missing)}")
if len(remaining_missing) > 0:
    print(remaining_missing.sort_values(ascending=False))
else:
    print("No missing values remain!")

Columns with remaining missing values: 13
median_income                                8217
housing_value                                8103
median_monthly_housing_costs                 7829
average_household_size                       5902
total_population                             4654
total_housing_units                          4654
educational_attainment_bachelor_or_higher    4654
poverty_status                               4654
housing_occupancy_number                     4654
year_structure_built                         4654
housing_vacancy_number                       4654
owner_occupied_housing_units                 4654
renter_occupied_housing_units                4654
dtype: int64


In [6]:
# Fill missing values
# Numeric columns: fill with median
# Categorical/boolean columns: fill with mode
for col in df.columns:
    if df[col].isna().sum() == 0:
        continue
    if df[col].dtype in ["float64", "int64"]:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])

# Verify no missing values remain
assert df.isna().sum().sum() == 0, "Still have missing values!"
print("All missing values handled. No NaN remaining.")
print(f"Shape: {df.shape}")

All missing values handled. No NaN remaining.
Shape: (47033, 48)


## Step 5: Balance Classes via Undersampling

In [7]:
# Undersample majority class
# Goal: ~2000 fires (yes) and ~4000 not fires (no) — roughly 2:1 ratio

df_yes = df[df["fire_occurred"] == "yes"]
df_no = df[df["fire_occurred"] == "no"]

print(f"Before undersampling:")
print(f"  yes (fire): {len(df_yes)}")
print(f"  no (no fire): {len(df_no)}")

# Keep all "yes" rows, undersample "no" to 2x the "yes" count
target_no_count = min(len(df_no), len(df_yes) * 2)
df_no_sampled = df_no.sample(n=target_no_count, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([df_yes, df_no_sampled]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter undersampling:")
print(df_balanced["fire_occurred"].value_counts())

Before undersampling:
  yes (fire): 11506
  no (no fire): 35527

After undersampling:
fire_occurred
no     23012
yes    11506
Name: count, dtype: int64


## Step 6: Save Cleaned Data

In [8]:
# Save cleaned dataset
df_balanced.to_csv("../data/cleaned/insurance_cleaned.csv", index=False)

print(f"Saved cleaned data to data/cleaned/insurance_cleaned.csv")
print(f"Final shape: {df_balanced.shape}")
print(f"\nColumn dtypes:")
print(df_balanced.dtypes)
print(f"\nTarget distribution:")
print(df_balanced["fire_occurred"].value_counts())
print(f"\nSummary stats:")
df_balanced.describe()

Saved cleaned data to data/cleaned/insurance_cleaned.csv
Final shape: (34518, 48)

Column dtypes:
Year                                           int64
ZIP                                            int64
Avg Fire Risk Score                          float64
CAT Cov A Fire -  Incurred Losses              int64
CAT Cov A Fire -  Number of Claims             int64
CAT Cov A Smoke -  Incurred Losses             int64
CAT Cov A Smoke -  Number of Claims            int64
CAT Cov C Fire -  Incurred Losses              int64
CAT Cov C Fire -  Number of Claims             int64
CAT Cov C Smoke -  Incurred Losses             int64
CAT Cov C Smoke -  Number of Claims            int64
Cov A Amount Weighted Avg                    float64
Cov C Amount Weighted Avg                    float64
Earned Exposure                              float64
Earned Premium                                 int64
Non-CAT Cov A Fire -  Incurred Losses          int64
Non-CAT Cov A Fire -  Number of Claims         int64
N

,Year,ZIP,Avg Fire Risk Score,CAT Cov A Fire - Incurred Losses,CAT Cov A Fire - Number of Claims,CAT Cov A Smoke - Incurred Losses,CAT Cov A Smoke - Number of Claims,CAT Cov C Fire - Incurred Losses,CAT Cov C Fire - Number of Claims,CAT Cov C Smoke - Incurred Losses,...,average_household_size,educational_attainment_bachelor_or_higher,poverty_status,housing_occupancy_number,housing_value,year_structure_built,housing_vacancy_number,median_monthly_housing_costs,owner_occupied_housing_units,renter_occupied_housing_units
count,34518.000000,34518.000000,34518.00000,3.451800e+04,34518.000000,3.451800e+04,34518.000000,3.451800e+04,34518.00000,3.451800e+04,...,34518.000000,34518.000000,34518.000000,34518.000000,3.451800e+04,34518.000000,34518.000000,34518.000000,34518.000000,34518.000000
mean,2019.648966,93626.249435,0.82819,2.114910e+05,0.891651,6.058327e+03,0.429921,7.216098e+04,0.73185,1.257023e+03,...,2.770165,3560.811895,2777.387798,8869.867258,7.100441e+05,8869.867258,657.905672,1933.786343,4627.575004,3491.424213
std,1.106198,1781.372146,0.87997,9.905038e+06,26.116508,1.117192e+05,5.630479,4.117645e+06,23.49847,3.591651e+04,...,0.618848,3699.817268,3278.245701,7596.056951,4.081123e+05,7596.056951,855.510733,698.269023,4252.348861,3827.104392
min,2018.000000,90001.000000,-0.08000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.00000,0.000000e+00,...,1.100000,0.000000,0.000000,0.000000,9.999000e+03,0.000000,0.000000,175.000000,0.000000,0.000000
25%,2019.000000,92263.000000,0.16000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.00000,0.000000e+00,...,2.420000,327.000000,353.000000,1495.000000,4.275000e+05,1495.000000,194.000000,1473.000000,731.000000,311.000000
50%,2020.000000,93603.000000,0.51500,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.00000,0.000000e+00,...,2.700000,2301.000000,1572.000000,7050.000000,6.235000e+05,7050.000000,436.000000,1884.000000,3471.000000,2068.000000
75%,2021.000000,95351.000000,1.15000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,0.00000,0.000000e+00,...,3.070000,5632.250000,4014.000000,14362.000000,8.508000e+05,14362.000000,818.000000,2315.000000,7192.000000,5490.000000
max,2021.000000,96162.000000,4.00000,1.616443e+09,4025.000000,7.458821e+06,458.000000,7.008890e+08,3697.00000,2.936536e+06,...,10.510000,18629.000000,27077.000000,39043.000000,2.000001e+06,39043.000000,10903.000000,4001.000000,23263.000000,26124.000000
